# 3. Demo - Imagette time series

Learning goals:
- Create a simple stellar catalogue
- Placing the simulated subfield
- Running a imagette time series
- Cautions and assumptions

### Setup notebook

In [14]:
# Reload code outside notebook
%load_ext autoreload
%autoreload 2

# Configure figures in notebook
%matplotlib notebook

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Imports

In [18]:
import os
import numpy as np
import matplotlib.pyplot as plt

# PlatoSim
from platosim.simulation   import Simulation
from platosim.simfile      import SimFile
from platosim.utilities    import getFunctions
from platosim.matplotlibrc import setup_notebook
setup_notebook()

### Define variables

In [16]:
homeDir = os.getenv('PLATO_PROJECT_HOME')
workDir = os.getenv('PLATO_WORKDIR')

## Exercise 3.1 - Create a stellar catalogue

Besides reading in a existing stellar catalogue, PlatoSim also allows a few options to generate custom stellar catalogues. First let's have a look at how you can create a stellar catalogue for a custom subfield. You can either create a catalogue from the pixel coordinates using the utility `sim.createStarCatalogFileFromPixelCoordinates()` or from a set of equatorial sky coordinates using the utility `sim.createStarCatalogFile()`.  

Let's have a look at the first method: 
- <font color='orange'>Inspect the docstring of the function `sim.createStarCatalogFileFromPixelCoordinates()`</font>
- <font color='orange'>Define an array for `row`, `col`, `mag` and `starID` each with three values (i.e. three stars of your choice)</font>
- <font color='orange'>Seen from the docstring, you need to specify the pixel coordinates (row, col) of the CCD, and not of the subfield. Hence, add the subfield zero-point (row, col) to your coordinates </font>
- <font color='orange'>Select a filename and create a stellar catalogue</font>


<font color='red'>Update function above to also take relative paths -> use `pathlib`</font>

In [21]:
sim = Simulation('test')
print(sim.createStarCatalogFileFromPixelCoordinates.__doc__)

Create a star catalogue file from the pixel coordinates.
        
        Create a star catalog ascii file given the pixel coordinates 
        (row and column) of the stars. This requires the orientation
        of the spacecraft, telescopes, focal plane, hence it's a member
        function of the Simulation class.
        
        Paramters
        ---------
        rows : ndarray
            Array with fractional row coordinates of the stars (CCD, not subfield) [pix]
        cols : ndarray
            Array with fractional column coordinates of the stars (CCD, not subfield) [pix]
        magnitudes : ndarray
            Array with Johnson V magnitudes of the stars
        starIDs : ndarray
            Array with IDs of the star (integers)
        starCatalogPath : str
            Path of the star catalog file that will be written.

        Return
        ------
        A file will be saved, containing, ra, dec, and magnitude of the stars.
        The "ObservingParameters/StarCatalo

Note that `sim.createStarCatalogFileFromPixelCoordinates()` automatically sets the path in the YAML tree to the newly created stellar catalogue file. If you need to tell PlatoSim to point to a different file simply use:
```
sim["ObservingParameters/StarCatalogFile"] = <starcatFile>
```

Let's have a look at your new catalogue:
- <font color='orange'>Run a new simulation (now with your catalogue set for use)</font>
- <font color='orange'>Visualise the subfield to check your input catalogue</font>

## Exercise 3.2 - Set a subfield around a set of coordinates

Instead of generating a star catalogue that fits into your subfield, it is also possible to create a star catalogue and place the subfield at their location. This is especially useful when you want to test features at a specific location on the CCD or if you want to create a grid/pattern of stars on the PLATO CCDs. Methods that we will focus on are:

* `pixelToSkyCoordinates()` with `sim.setSubfieldAroundSkyCoordinates()`
* `skyToPixelCoordinates()` with `sim.setSubfieldAroundPixelCoordinates()`

These methods combine the above steps in one simple function and take the field distortion into account. Both of these methods include functions from the `refereceFrames` script:

In [17]:
import platosim.referenceFrames as rf
getFunctions(rf)

platosim.referenceFrames functions
calculateSubfieldAroundCoordinates
computeCCDcornersInFocalPlane
distortedToUndistortedFocalPlaneCoordinates
ecliptic2equatorial
eprint
equatorial2ecliptic
focalPlaneAngles2pixelCoordinates
focalPlaneCoordinatesFromGnomonicRadialDistance
focalPlaneToPixelCoordinates
focalPlaneToSkyCoordinates


In the first exercise you will use `rf.pixelToSkyCoordinates()` together with `sim.setSubfieldAroundSkyCoordinates()`. The latter function place the subfield given a set of equatorial sky coordinates (RA, Dec). Note that if the subfield falls outside the camera FOV or do not fall on any of the four CCDs, then this function will return `False`.

- <font color='orange'>Define a subfield using the parameters (`ccdCode`, `xCCDpixel`, `yCCDpixel`, `xSubfieldSize` `ySubfieldSize`)</font>
- <font color='orange'>Inspect the docstring of `rf.pixelToSkyCoordinates()` and fetch the sky coordinates corresponding to the center of the subfield (`raCenter`, `decCenter`)</font>
- <font color='orange'>Inspect the docstring of `sim.setSubfieldAroundSkyCoordinates()` and use it to place the subfield around your sky coordinates</font>
- <font color='orange'>Run the simulation and visualise the subfield</font>

Now we will try to use `rf.skyToPixelCoordinates()` together with `sim.setSubfieldAroundPixelCoordinates()` to recreate the subfield from above:

- <font color='orange'>Inspect the docstring of `rf.skyToPixelCoordinates()` and use the sky coordinates (`raCenter`, `decCenter`) from above to get the central pixel coordinates of your subfield</font>
- <font color='orange'>Inspect the docstring of `sim.setSubfieldAroundPixelCoordinates()` and use it to place the subfield around your pixel coordinates</font>
- <font color='orange'>Run the simulation and visualise the subfield (does it match the above simulation?)</font>

## Exercise 3.3 - Imagette time series

It's time to put the knowledge up until now into practise and generate an imagette time series. During a mission quarter the thermal profile between the optical bench and each camera is subject to change - this is called Thermo-Elastic Distortion (TED). This mean that in a worst case scenario a single camera can "drift" away from it's original pointing by an angle of up to 10 arcsec in 91 days. Combined with the kinematic aberration this results in an worst case drift of 15 arcsec in 91 days. Note PlatoSim by default includes a realistic orbit model (from CreMa) that takes into account the differential kinematic aberration, however, to be sure that we reach the worst case scenario, we will turn it off an only consider the TED. Also just to speed up things, we will only simulate 10 days of data here. 

- <font color='orange'>Create a new simulation object</font>
- <font color='orange'>Create a star catalogue using one of the methods from above (use at least 5 stars)</font>
- <font color='orange'>Change the subfield's dimensions to that of an imagette</font>
- <font color='orange'>Change the number of exposures to be equal to a duration of 10 days</font>
- <font color='orange'>Turn off the (differential) kinematic aberration</font>
- <font color='orange'>Inspect the docstring of the function `sim.createDriftFile()` and create TED file containing a linear drift in (yaw, pitch, roll) of each 15 arcsec over a duration of 10 days</font>
- <font color='orange'>Save only the pixel maps and the stellar coordinates</font>
- <font color='orange'>Run and visualise the simulation: do you see anything strange when using the slider?</font>

<font color='orange'>**Extra exercise:**</font> 

- <font color='orange'>PlatoSim also includes a model for electronic and digital saturation. Setup a subfield of 200 x 200 pixels and create a star catalogue where the stars are placed equidistantly distributed along the pixel column. Use a range of stellar magnitudes spanning 5-9 and run a simulation of 1 exposure. Where does saturation kick in for the N-CAMs?</font>
- <font color='orange'>Similarly, setup a semilar simulation and try to see to what limit PLATO is magnitude limited</font>